# 0002 — IBOV point-in-time vs survivors

| | |
|---|---|
| **Hypothesis** | Choosing the universe with hindsight (today's IBOV applied to the past) inflates results. The size of that inflation on B3 is unknown. |
| **Universe** | (1) IBOV members of each period, point-in-time; (2) today's IBOV members over the whole past; (3) BOVA11 as benchmark from its listing. |
| **Period** | First IBOV period of 2003 → last session of the base; daily; equal weight at each period start, held to the next. |
| **Result** | _Fill in after running._ |
| **Conclusion** | _Fill in after running._ |
| **Drove in `src/`** | _Fill in: what this notebook showed is missing from the library._ |

## Setup

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from ibexQuant.analytics import B3_SESSIONS_PER_YEAR, performance_summary
from ibexQuant.data import B3Dataset
from ibexQuant.data.universe import Index, Tickers, Universe

In [ ]:
START = "2003-01-01"
INDEX = "IBOV"
BENCHMARK = "BOVA11"

pd.set_option("display.precision", 4)

ds = B3Dataset.open()
END = ds.calendar.sessions[-1]
print(f"base: {ds.calendar.sessions[0].date()} → {END.date()}")

## Universes

Rebalancing happens at the close of the **first session of each IBOV period**: the members
of the new portfolio are bought in equal weight and held, drifting with prices, until the
first session of the next period.

- **Point-in-time** uses `Index("IBOV")`: the members of each period, from B3's portfolios,
  the retroactive series (2003–2014) and the liquidity rule where B3 has none (2014-05 to
  2017-05).
- **Survivors** uses today's members, `Tickers([...])`, over the whole past: each one enters
  as soon as it is listed. This is the universe a backtest gets by picking stocks today.

In [ ]:
rebalances = ds.rebalance_dates(INDEX, START, END)
edges = [*rebalances, END]

today = ds.constituents(INDEX, END)
survivors = today["ticker"][today["security_id"].notna()].tolist()

print(f"{len(rebalances)} periods, {rebalances[0].date()} → {rebalances[-1].date()}")
print(f"today's {INDEX}: {len(survivors)} members (period {today['period'].iloc[0]})")

In [ ]:
# Where each period's composition comes from.
sources = pd.Series(
    [ds.constituents(INDEX, day)["source"].iloc[0] for day in rebalances], index=rebalances
)
sources.value_counts()

## Mechanics

One `ds.panel(...)` per period, from its first session to the first session of the next,
adjusted (`total_return`) to the end of the window: inside the window every return includes
the events, as a holder reinvesting them would see.

A member that stops trading inside the period (`ds.exits`) keeps the value of its last trade
until the next rebalance. That is a placeholder for pending item P10 (delisting returns),
not a model of it.

In [ ]:
def equal_weight(universe: Universe, label: str) -> tuple[pd.Series, pd.DataFrame]:
    """Daily returns of an equal-weight portfolio rebalanced at each period start."""
    pieces: list[pd.Series] = []
    log: list[dict[str, object]] = []
    for entry, exit_ in zip(edges[:-1], edges[1:], strict=True):
        if entry >= exit_:
            continue
        panel = ds.panel(universe, entry, exit_)
        close = panel.field("close")
        held = close.columns[panel.universe_mask.loc[entry] & close.loc[entry].notna()]
        # ffill: a gap or an exit keeps the last traded value (P10 placeholder).
        value = (close[held].ffill() / close.loc[entry, held]).mean(axis=1)
        pieces.append(value.pct_change().iloc[1:])
        log.append({"entry": entry, "members": len(held), "window_return": value.iloc[-1] - 1})
    return pd.concat(pieces).rename(label), pd.DataFrame(log).set_index("entry")


started = time.perf_counter()
point_in_time, pit_log = equal_weight(Index(INDEX), "point_in_time")
survivor_returns, survivor_log = equal_weight(Tickers(survivors), "survivors")
print(f"{2 * len(rebalances)} panels in {time.perf_counter() - started:.0f} s")

In [ ]:
benchmark_close = ds.panel([BENCHMARK], rebalances[0], END).field("close").iloc[:, 0]
benchmark = benchmark_close.pct_change().dropna().rename("BOVA11")
print(f"{BENCHMARK} from {benchmark.index[0].date()}")

In [ ]:
# Members held per period, and exits inside each period.
exits = ds.exits(INDEX, rebalances[0], END).groupby("period").size()
periods = pit_log.assign(
    survivors=survivor_log["members"],
    period=[day.strftime("%Y-%m") for day in pit_log.index],
)
periods["exits"] = periods["period"].map(exits).fillna(0).astype(int)
periods["source"] = sources
periods[["period", "source", "members", "survivors", "exits", "window_return"]].tail(12)

## Metrics

In [ ]:
returns = pd.concat([point_in_time, survivor_returns], axis=1)
full = performance_summary(returns, periods_per_year=B3_SESSIONS_PER_YEAR)
full

In [ ]:
# From BOVA11's listing, against the benchmark.
recent = performance_summary(
    returns.loc[benchmark.index[0] :],
    benchmark=benchmark,
    periods_per_year=B3_SESSIONS_PER_YEAR,
)
recent

In [ ]:
years = (returns.index[-1] - returns.index[0]).days / 365.25
growth = (1 + returns).prod()
cagr = growth ** (1 / years) - 1
gap = cagr["survivors"] - cagr["point_in_time"]
print(f"CAGR point-in-time {cagr['point_in_time']:.2%}, survivors {cagr['survivors']:.2%}")
multiple = growth["survivors"] / growth["point_in_time"]
print(f"survivorship gap: {gap:.2%} a year, {multiple:.1f}x the final value")

## Charts

In [ ]:
equity = (1 + returns).cumprod()
fig, (ax_equity, ax_gap) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, height_ratios=[2, 1])

equity.plot(ax=ax_equity, logy=True, title="Growth of 1 (log scale)")
bench_equity = (1 + benchmark).cumprod() * equity.loc[benchmark.index[0] :, "point_in_time"].iloc[
    0
]
bench_equity.plot(ax=ax_equity, color="gray", linestyle="--", label="BOVA11")
ax_equity.legend()
ax_equity.grid(alpha=0.3)

rolling = (1 + returns).rolling(B3_SESSIONS_PER_YEAR).apply(lambda r: r.prod(), raw=True) - 1
(rolling["survivors"] - rolling["point_in_time"]).plot(
    ax=ax_gap, title="Survivors minus point-in-time, trailing 12 months"
)
ax_gap.axhline(0, color="black", linewidth=0.8)
ax_gap.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
ax = periods.set_index("period")[["members", "survivors"]].plot(
    figsize=(12, 4), title="Members held at each rebalance"
)
ax.grid(alpha=0.3)
plt.tight_layout()

## Known simplifications

- **No costs, no lot size, fractional shares.** Every rebalance trades the whole portfolio.
- **Zero risk-free rate.** Sharpe ignores the CDI.
- **Adjusted prices, gross JCP.** Events are reinvested inside each asset (P9 is the
  engine's alternative: raw prices and cash).
- **Exits keep their last value** until the next rebalance (P10).
- **Composition sources differ by era**: retroactive series before 2014, liquidity rule
  2014-05 to 2017-05, B3's previews after. The point-in-time line inherits their quality.
- **Equal weight, not index weight.** Neither line is the IBOV itself; BOVA11 is shown for
  scale, with its fee and tracking error.